# DS4200 — Whale Wallets vs Global Events
**Research Question:** What has a bigger impact on crypto — whale wallets or global events?

---

In [1]:
import pandas as pd
import numpy as np
import altair as alt
import json
from IPython.display import HTML

# Inline data — no external JSON files (required for VS Code notebook renderer)
alt.data_transformers.enable("default", max_rows=None)
# Native Vega-Lite MIME type — renders in VS Code without CDN
alt.renderers.enable("mimetype")

print("Libraries loaded OK")

Libraries loaded OK


## Data Loading

In [2]:
cg  = pd.read_csv("../data/processed/coingecko_processed.csv",   parse_dates=["date"])
bh  = pd.read_csv("../data/processed/btc_history_processed.csv", parse_dates=["date"])

btc_cg = cg[cg["coin"] == "bitcoin"].copy().reset_index(drop=True)
eth_cg = cg[cg["coin"] == "ethereum"].copy().reset_index(drop=True)
bh     = bh.dropna(subset=["price_change_pct"]).copy()

print(f"BTC (CoinGecko): {btc_cg.date.min().date()} → {btc_cg.date.max().date()}  ({len(btc_cg)} rows)")
print(f"ETH (CoinGecko): {eth_cg.date.min().date()} → {eth_cg.date.max().date()}  ({len(eth_cg)} rows)")
print(f"BTC On-Chain:    {bh.date.min().date()} → {bh.date.max().date()}  ({len(bh)} rows)")

BTC (CoinGecko): 2025-02-19 → 2026-02-18  (366 rows)
ETH (CoinGecko): 2025-02-19 → 2026-02-18  (366 rows)
BTC On-Chain:    2024-02-20 → 2026-02-18  (730 rows)


## Event Definitions & BTC Impact Computation

In [3]:
# ── Event list (synced with ETH wallet notebook) ──────────────────────────────
# Columns: date, label, category, lat, lon
RAW_EVENTS = [
    # Macro / Fed
    ("2025-05-07", "Fed Holds Rates",        "macro",        38.90, -77.03),
    ("2025-06-18", "Fed Holds Rates",        "macro",        38.88, -76.98),
    ("2025-07-30", "Fed Holds Rates",        "macro",        38.92, -77.06),
    ("2025-09-17", "Fed Cuts -0.25%",        "macro",        38.86, -77.01),
    ("2025-11-07", "Fed Holds Rates",        "macro",        38.91, -77.04),
    ("2025-12-10", "Fed Holds Rates",        "macro",        38.89, -76.97),
    ("2026-01-28", "Fed Holds Rates",        "macro",        38.87, -77.02),
    # Regulatory
    ("2025-04-10", "SEC Crypto Roundtable",  "regulatory",   38.89, -77.05),
    ("2025-05-12", "Stablecoin Bill Debate", "regulatory",   38.88, -77.03),
    ("2025-07-14", "US Crypto Framework",    "regulatory",   38.87, -77.01),
    # Crypto-native
    ("2025-04-14", "ETH Pectra Upgrade",     "crypto",       47.20,   8.54),  # Zug, Switzerland
    ("2025-05-01", "BTC $100k Reclaimed",    "crypto",       25.77, -80.19),  # Miami
    ("2025-11-03", "ETH ATH Retested",       "crypto",       40.71, -74.01),  # New York
    # Geopolitical
    ("2025-04-09", "Tariff Pause 90d",       "geopolitical", 38.90, -77.10),
    ("2025-05-22", "US-China Trade Talks",   "geopolitical", 46.20,   6.15),  # Geneva
    ("2025-07-09", "Tariff Deadline",        "geopolitical", 38.85, -77.05),
    ("2026-01-15", "BRICS Summit",           "geopolitical",-22.91, -43.17),  # Rio de Janeiro
]

events_df = pd.DataFrame(RAW_EVENTS, columns=["date","label","category","lat","lon"])
events_df["date"] = pd.to_datetime(events_df["date"])

EVENT_COLORS = {
    "macro":        "#f7931a",
    "regulatory":   "#ab47bc",
    "crypto":       "#26a69a",
    "geopolitical": "#ef5350",
}
events_df["color"] = events_df["category"].map(EVENT_COLORS)

# ── Compute BTC 3-day price impact per event ───────────────────────────────────
btc_px  = btc_cg.set_index("date")["price"]
bh_px   = bh.set_index("date")["market_price_usd"]
all_btc = pd.concat([btc_px, bh_px[~bh_px.index.isin(btc_px.index)]]).sort_index()

def btc_impact(event_date, window=3):
    before = all_btc[all_btc.index <= event_date]
    after  = all_btc[all_btc.index >  event_date]
    if len(before) == 0 or len(after) == 0:
        return np.nan
    p0 = before.iloc[-1]
    p1 = after.iloc[min(window - 1, len(after) - 1)]
    return ((p1 - p0) / p0) * 100

events_df["btc_impact_pct"] = events_df["date"].apply(btc_impact)
events_df = events_df.dropna(subset=["btc_impact_pct"]).reset_index(drop=True)

# ── Whale proxy stats from BTC on-chain data ──────────────────────────────────
bh_sorted         = bh.sort_values("date").copy()
bh_sorted["next_day_pct"] = bh_sorted["price_change_pct"].shift(-1)
top5_whale        = bh.nlargest(5, "avg_tx_value_usd").reset_index()
valid             = bh_sorted.dropna(subset=["avg_tx_value_usd","next_day_pct"])
pearson_r         = valid["avg_tx_value_usd"].corr(valid["next_day_pct"])

print(f"Events computed: {len(events_df)}")
print(f"BTC impact range: {events_df.btc_impact_pct.min():.2f}% → {events_df.btc_impact_pct.max():.2f}%")
print(f"Whale proxy Pearson r (avg_tx_value vs next-day return): {pearson_r:.4f}")
events_df[["date","label","category","btc_impact_pct"]].sort_values("btc_impact_pct", ascending=False)

Events computed: 17
BTC impact range: -6.12% → 9.32%
Whale proxy Pearson r (avg_tx_value vs next-day return): -0.0292


,date,label,category,btc_impact_pct
13,2025-04-09,Tariff Pause 90d,geopolitical,9.315187
15,2025-07-09,Tariff Deadline,geopolitical,7.909666
0,2025-05-07,Fed Holds Rates,macro,6.306373
4,2025-11-07,Fed Holds Rates,macro,3.342825
7,2025-04-10,SEC Crypto Roundtable,regulatory,3.247224
11,2025-05-01,BTC $100k Reclaimed,crypto,1.790313
10,2025-04-14,ETH Pectra Upgrade,crypto,0.604012
9,2025-07-14,US Crypto Framework,regulatory,-0.310109
8,2025-05-12,Stablecoin Bill Debate,regulatory,-0.384287
3,2025-09-17,Fed Cuts -0.25%,macro,-0.948109


---
## Chart 1 — Global Event Impact Map (D3.js)
*World map: circles at each event origin, sized by BTC 3-day price impact.*

In [4]:
# Chart 1 — Altair geographic map (renders natively in VS Code via vnd.vegalite.v5+json)
world_topo = alt.topo_feature(
    "https://cdn.jsdelivr.net/npm/world-atlas@2/countries-110m.json",
    "countries"
)

# World background
background = alt.Chart(world_topo).mark_geoshape(
    fill="#1c2333",
    stroke="#2d3a50",
    strokeWidth=0.5
)

# Prep events data
ev_map = events_df.copy()
ev_map["abs_impact"] = ev_map["btc_impact_pct"].abs()
ev_map["impact_label"] = ev_map["btc_impact_pct"].apply(lambda x: f"{x:+.2f}%")
ev_map["date_str"] = ev_map["date"].dt.strftime("%Y-%m-%d")

cat_colors = alt.Scale(
    domain=["macro", "regulatory", "crypto", "geopolitical"],
    range=["#f7931a", "#ab47bc", "#26a69a", "#ef5350"]
)

dir_colors = alt.Scale(
    domain=[True, False],
    range=["rgba(38,166,154,0.75)", "rgba(239,83,80,0.75)"]
)
ev_map["positive"] = ev_map["btc_impact_pct"] >= 0

points = alt.Chart(ev_map).mark_point(filled=True, opacity=0.85).encode(
    longitude="lon:Q",
    latitude="lat:Q",
    size=alt.Size("abs_impact:Q",
                  scale=alt.Scale(range=[60, 1400]),
                  legend=alt.Legend(title="|BTC Δ| (%)", orient="bottom-right",
                                    labelColor="white", titleColor="white")),
    color=alt.Color("positive:N",
                    scale=dir_colors,
                    legend=None),
    stroke=alt.Color("category:N",
                     scale=cat_colors,
                     legend=alt.Legend(title="Event Type", orient="bottom-left",
                                       labelColor="white", titleColor="white")),
    strokeWidth=alt.value(2.5),
    tooltip=[
        alt.Tooltip("label:N", title="Event"),
        alt.Tooltip("date_str:N", title="Date"),
        alt.Tooltip("category:N", title="Category"),
        alt.Tooltip("impact_label:N", title="BTC 3-day Impact"),
    ]
)

(background + points).project(
    type="naturalEarth1"
).properties(
    width=880, height=460,
    title=alt.TitleParams(
        "Global Events & BTC 3-Day Price Impact",
        subtitle=["Fill = BTC direction (teal=positive, red=negative)  |  Border color = event type  |  Size = magnitude"],
        color="white", subtitleColor="#888", fontSize=16, subtitleFontSize=11
    )
).configure(
    background="#111111"
).configure_view(
    strokeOpacity=0
).configure_title(
    color="white", subtitleColor="#888"
)

/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/c

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


---
## Chart 2 — ETH Price + Global Events + Volume (Altair)
*ETH price line with event rule markers, large swing diamonds, and volume panel.*

In [5]:
# Filter events to the ETH data date range
eth_min, eth_max = eth_cg["date"].min(), eth_cg["date"].max()
ev_eth = events_df[(events_df["date"] >= eth_min) & (events_df["date"] <= eth_max)].copy()
ev_eth["cat_label"] = ev_eth["category"].map({
    "macro":"Macro / Fed","regulatory":"Regulatory",
    "crypto":"Crypto-Native","geopolitical":"Geopolitical"
})

CAT_DOMAIN  = ["Macro / Fed","Regulatory","Crypto-Native","Geopolitical"]
CAT_RANGE   = ["#f7931a",    "#ab47bc",   "#26a69a",      "#ef5350"]
cat_scale   = alt.Scale(domain=CAT_DOMAIN, range=CAT_RANGE)
cat_legend  = alt.Legend(title="Event Category", titleColor="white",
                          labelColor="white", orient="top-left")

In [6]:
eth_line = alt.Chart(eth_cg).mark_line(color="#627eea", strokeWidth=2.5).encode(
    x=alt.X("date:T", title=None),
    y=alt.Y("price:Q", title="ETH Price (USD)", axis=alt.Axis(format="$~s")),
    tooltip=[alt.Tooltip("date:T"), alt.Tooltip("price:Q", title="ETH Price", format="$,.0f"),
             alt.Tooltip("daily_return:Q", title="Return %", format=".2f")]
)

ma30 = alt.Chart(eth_cg).mark_line(strokeDash=[6,3], strokeWidth=1.5, opacity=0.55,
                                    color="#f7931a").encode(
    x="date:T", y="price_30d_avg:Q"
)

swings = alt.Chart(eth_cg[eth_cg["large_swing"]]).mark_point(
    shape="diamond", size=130, filled=True, opacity=0.95
).encode(
    x="date:T", y="price:Q",
    color=alt.condition(alt.datum.daily_return > 0, alt.value("#26a69a"), alt.value("#ef5350")),
    tooltip=[alt.Tooltip("date:T"), alt.Tooltip("price:Q", format="$,.0f"),
             alt.Tooltip("daily_return:Q", title="Return %", format="+.2f")]
)

ev_rules = alt.Chart(ev_eth).mark_rule(strokeWidth=1.8, opacity=0.65).encode(
    x="date:T",
    color=alt.Color("cat_label:N", scale=cat_scale, legend=cat_legend),
    tooltip=[alt.Tooltip("date:T", title="Event Date"), alt.Tooltip("label:N", title="Event"),
             alt.Tooltip("cat_label:N", title="Category"),
             alt.Tooltip("btc_impact_pct:Q", title="BTC 3d Impact %", format="+.2f")]
)

vol_bars = alt.Chart(eth_cg).mark_bar(opacity=0.8).encode(
    x=alt.X("date:T", title="Date"),
    y=alt.Y("volume:Q", title="ETH Volume (USD)", axis=alt.Axis(format="$~s")),
    color=alt.condition(alt.datum.high_volume, alt.value("#f7931a"), alt.value("#2a2a4a")),
    tooltip=[alt.Tooltip("date:T"), alt.Tooltip("volume:Q", format="$,.0f"),
             alt.Tooltip("high_volume:N", title="High Volume Day")]
)

price_panel = (eth_line + ma30 + swings + ev_rules).properties(
    width=820, height=300,
    title=alt.TitleParams(
        "Ethereum Price + Global Events + Large Swing Days",
        subtitle=["◆ = large swing day (green up / red down)  |  Dashed = 30d MA  |  Vertical lines = global events"],
        color="white", subtitleColor="#888", fontSize=16
    )
)

vol_panel = (vol_bars + ev_rules.mark_rule(opacity=0.35, strokeWidth=1.2)).properties(
    width=820, height=140,
    title=alt.TitleParams("ETH Volume  (Orange = High-Volume / Potential Whale Activity Days)",
                          color="#aaa", fontSize=12)
)

alt.vconcat(price_panel, vol_panel, spacing=8).configure(
    background="#1a1a1a"
).configure_axis(
    labelColor="white", titleColor="white", gridColor="#252525"
).configure_legend(
    labelColor="white", titleColor="white", fillColor="#222", strokeColor="#444",
    padding=8, cornerRadius=6
).configure_title(color="white")

/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/c

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


---
## Chart 3 — Correlation + Top 5 Events vs Top 5 Whale Days (Altair)

In [7]:
# ── Prep: Top 5 events + top 5 whale days ─────────────────────────────────────
top5_ev = (events_df
    .assign(abs_impact=events_df["btc_impact_pct"].abs())
    .nlargest(5, "abs_impact")
    .sort_values("btc_impact_pct")
    .copy()
    .reset_index(drop=True)
)
top5_ev["direction"] = top5_ev["btc_impact_pct"].apply(lambda x: "Positive" if x >= 0 else "Negative")
top5_ev["ev_label"]  = top5_ev["label"] + "  (" + top5_ev["date"].dt.strftime("%b %Y") + ")"

top5_w = top5_whale.copy()
top5_w["date"]      = pd.to_datetime(top5_w["date"])
top5_w["direction"] = top5_w["price_change_pct"].apply(lambda x: "Positive" if x >= 0 else "Negative")
top5_w["wh_label"]  = top5_w["date"].dt.strftime("%b %d '%y") + "  ($" + \
                       (top5_w["avg_tx_value_usd"]/1000).round(0).astype(int).astype(str) + "k avg)"

print("Top 5 global events by BTC impact:")
print(top5_ev[["ev_label","btc_impact_pct"]].to_string(index=False))
print("\nTop 5 whale activity days:")
print(top5_w[["wh_label","avg_tx_value_usd","price_change_pct"]].to_string(index=False))

Top 5 global events by BTC impact:
                    ev_label  btc_impact_pct
ETH ATH Retested  (Nov 2025)       -6.120413
 Fed Holds Rates  (Jan 2026)       -5.675117
 Fed Holds Rates  (May 2025)        6.306373
 Tariff Deadline  (Jul 2025)        7.909666
Tariff Pause 90d  (Apr 2025)        9.315187

Top 5 whale activity days:
               wh_label  avg_tx_value_usd  price_change_pct
Nov 22 '25  ($689k avg)     689189.836011           -1.5848
May 28 '24  ($368k avg)     367648.179569            1.2961
Feb 05 '26  ($309k avg)     309372.639920           -3.4740
Jul 05 '24  ($304k avg)     304202.899214           -5.1737
Nov 12 '24  ($299k avg)     299110.997978           10.2412


In [8]:
clr_scale = alt.Scale(domain=["Positive","Negative"], range=["#26a69a","#ef5350"])

# ── Left: Top 5 events bar ────────────────────────────────────────────────────
ev_bar = alt.Chart(top5_ev).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4).encode(
    y=alt.Y("ev_label:N", title="", sort=alt.EncodingSortField("btc_impact_pct", order="ascending")),
    x=alt.X("btc_impact_pct:Q", title="BTC Price Change — 3-day window (%)",
            axis=alt.Axis(format="+.1f")),
    color=alt.Color("direction:N", scale=clr_scale, legend=None),
    tooltip=[alt.Tooltip("label:N", title="Event"), alt.Tooltip("date:T"),
             alt.Tooltip("category:N"), alt.Tooltip("btc_impact_pct:Q", format="+.2f", title="BTC % Impact")]
)
ev_txt = alt.Chart(top5_ev).mark_text(align="left", dx=5, fontSize=11, fontWeight="bold").encode(
    y=alt.Y("ev_label:N", sort=alt.EncodingSortField("btc_impact_pct", order="ascending")),
    x="btc_impact_pct:Q",
    text=alt.Text("btc_impact_pct:Q", format="+.1f"),
    color=alt.Color("direction:N", scale=clr_scale, legend=None)
)
ev_chart = (ev_bar + ev_txt).properties(
    width=330, height=230,
    title=alt.TitleParams("Top 5 Global Events", subtitle=["by BTC 3-day price impact"],
                          color="white", subtitleColor="#888", fontSize=14)
)

# ── Right: Top 5 whale days bar ───────────────────────────────────────────────
wh_bar = alt.Chart(top5_w).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4).encode(
    y=alt.Y("wh_label:N", title="", sort=alt.EncodingSortField("avg_tx_value_usd", order="ascending")),
    x=alt.X("avg_tx_value_usd:Q", title="Avg Tx Value (USD) — Whale Proxy",
            axis=alt.Axis(format="$~s")),
    color=alt.Color("direction:N", scale=clr_scale, legend=None),
    tooltip=[alt.Tooltip("date:T"), alt.Tooltip("avg_tx_value_usd:Q", format="$,.0f", title="Avg Tx Value"),
             alt.Tooltip("price_change_pct:Q", format="+.2f", title="BTC Same-Day %"),
             alt.Tooltip("tx_count:Q", format=",", title="Tx Count")]
)
wh_txt = alt.Chart(top5_w).mark_text(align="left", dx=5, fontSize=11, fontWeight="bold").encode(
    y=alt.Y("wh_label:N", sort=alt.EncodingSortField("avg_tx_value_usd", order="ascending")),
    x="avg_tx_value_usd:Q",
    text=alt.Text("price_change_pct:Q", format="+.1f"),
    color=alt.Color("direction:N", scale=clr_scale, legend=None)
)
wh_chart = (wh_bar + wh_txt).properties(
    width=330, height=230,
    title=alt.TitleParams("Top 5 Whale Activity Days", subtitle=["bar = avg tx value  |  label = BTC same-day %"],
                          color="white", subtitleColor="#888", fontSize=14)
)

# ── Bottom: Correlation scatter ────────────────────────────────────────────────
scatter = alt.Chart(valid).mark_circle(opacity=0.4, size=35).encode(
    x=alt.X("avg_tx_value_usd:Q", title="Avg Tx Value USD (Whale Proxy) — Log Scale",
            scale=alt.Scale(type="log"), axis=alt.Axis(format="$~s")),
    y=alt.Y("next_day_pct:Q", title="Next-Day BTC Price Change (%)"),
    color=alt.condition(alt.datum.next_day_pct > 0, alt.value("#26a69a"), alt.value("#ef5350")),
    tooltip=[alt.Tooltip("date:T"), alt.Tooltip("avg_tx_value_usd:Q", format="$,.0f", title="Avg Tx Value"),
             alt.Tooltip("next_day_pct:Q", format="+.2f", title="Next-Day %")]
)
reg = scatter.transform_regression(
    "avg_tx_value_usd", "next_day_pct"
).mark_line(color="#f7931a", strokeWidth=2.5, opacity=0.85)

corr_plot = (scatter + reg).properties(
    width=700, height=210,
    title=alt.TitleParams(
        f"Whale Activity vs Next-Day BTC Return  |  Pearson r = {pearson_r:.3f}",
        subtitle=["Orange = regression line  |  Weak r → whale activity alone does not reliably predict price movement"],
        color="white", subtitleColor="#888", fontSize=14
    )
).interactive()

alt.vconcat(
    alt.hconcat(ev_chart, wh_chart, spacing=44),
    corr_plot,
    spacing=28
).configure(background="#1a1a1a").configure_axis(
    labelColor="white", titleColor="white", gridColor="#252525"
).configure_title(color="white")

/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/c

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


---
## Chart 4 — BTC On-Chain Volume + Whale Tiers + Global Events (Altair)

In [9]:
# ── Classify whale activity tiers ────────────────────────────────────────────
bh_plot = bh.copy()
bh_plot["whale_tier"] = pd.cut(
    bh_plot["avg_tx_value_usd"],
    bins=[0, 75_000, 150_000, 300_000, np.inf],
    labels=["Normal", "Elevated", "High", "Extreme"]
)

# Filter events to BTC history range
bh_min, bh_max = bh_plot["date"].min(), bh_plot["date"].max()
ev_bh = events_df[(events_df["date"] >= bh_min) & (events_df["date"] <= bh_max)].copy()
ev_bh["cat_label"] = ev_bh["category"].map({
    "macro":"Macro / Fed","regulatory":"Regulatory",
    "crypto":"Crypto-Native","geopolitical":"Geopolitical"
})

print(f"Whale tier distribution:\n{bh_plot['whale_tier'].value_counts().sort_index()}")

Whale tier distribution:
whale_tier
Normal      214
Elevated    371
High        137
Extreme       4
Name: count, dtype: int64


In [10]:
ev_rules_bh = alt.Chart(ev_bh).mark_rule(strokeWidth=2, opacity=0.7).encode(
    x="date:T",
    color=alt.Color("cat_label:N",
        scale=alt.Scale(domain=CAT_DOMAIN, range=CAT_RANGE),
        legend=alt.Legend(title="Event Category", titleColor="white", labelColor="white",
                          orient="top-right")),
    tooltip=[alt.Tooltip("date:T", title="Event"), alt.Tooltip("label:N"),
             alt.Tooltip("btc_impact_pct:Q", format="+.2f", title="BTC 3d Impact %")]
)

# ── Top panel: tx volume bars + extreme whale highlights + events ──────────────
vol_bars_bh = alt.Chart(bh_plot).mark_bar(opacity=0.8).encode(
    x=alt.X("date:T", title=None),
    y=alt.Y("tx_volume_usd:Q", title="BTC On-Chain Tx Volume (USD)", axis=alt.Axis(format="$~s")),
    color=alt.condition(alt.datum.price_change_pct > 0, alt.value("#26a69a"), alt.value("#ef5350")),
    tooltip=[alt.Tooltip("date:T"), alt.Tooltip("tx_volume_usd:Q", format="$,.0f", title="Tx Volume"),
             alt.Tooltip("price_change_pct:Q", format="+.2f", title="Price %"),
             alt.Tooltip("avg_tx_value_usd:Q", format="$,.0f", title="Avg Tx Value"),
             alt.Tooltip("whale_tier:N", title="Whale Tier")]
)

extreme_overlay = alt.Chart(bh_plot[bh_plot["whale_tier"] == "Extreme"]).mark_bar(
    color="#f7931a", opacity=0.95
).encode(
    x="date:T",
    y=alt.Y("tx_volume_usd:Q"),
    tooltip=[alt.Tooltip("date:T"), alt.Tooltip("avg_tx_value_usd:Q", format="$,.0f", title="Avg Tx (EXTREME)"),
             alt.Tooltip("price_change_pct:Q", format="+.2f")]
)

vol_panel_bh = (vol_bars_bh + extreme_overlay + ev_rules_bh).properties(
    width=820, height=280,
    title=alt.TitleParams(
        "BTC On-Chain Transaction Volume + Global Events Overlay",
        subtitle=["Green = positive day | Red = negative | Orange = Extreme whale-tier (avg tx > $300k) | Lines = events"],
        color="white", subtitleColor="#888", fontSize=16
    )
)

# ── Bottom panel: continuous whale signal + tier thresholds ───────────────────
whale_area = alt.Chart(bh_plot).mark_area(
    color="#9c27b0", opacity=0.4, line={"color":"#ce93d8","strokeWidth":1.8}
).encode(
    x=alt.X("date:T", title="Date"),
    y=alt.Y("avg_tx_value_usd:Q", title="Avg Tx Value (Whale Signal)", axis=alt.Axis(format="$~s")),
    tooltip=[alt.Tooltip("date:T"), alt.Tooltip("avg_tx_value_usd:Q", format="$,.0f", title="Avg Tx Value"),
             alt.Tooltip("whale_tier:N", title="Whale Tier")]
)

thresh_df = pd.DataFrame([
    {"y":75_000,  "label":"Elevated"},
    {"y":150_000, "label":"High"},
    {"y":300_000, "label":"Extreme"},
])
thresh_lines = alt.Chart(thresh_df).mark_rule(
    strokeDash=[3,3], strokeWidth=1, opacity=0.45, color="#777"
).encode(y="y:Q")
thresh_text = alt.Chart(thresh_df).mark_text(
    align="right", dx=-5, dy=-7, fontSize=9, color="#777"
).encode(y="y:Q", x=alt.value(820), text="label:N")

whale_panel_bh = (whale_area + ev_rules_bh.mark_rule(opacity=0.45, strokeWidth=1.2) + thresh_lines + thresh_text).properties(
    width=820, height=160,
    title=alt.TitleParams("Whale Activity Intensity  (Dashed thresholds = tier boundaries)",
                          color="#ce93d8", fontSize=12)
)

alt.vconcat(vol_panel_bh, whale_panel_bh, spacing=10).configure(
    background="#1a1a1a"
).configure_axis(
    labelColor="white", titleColor="white", gridColor="#252525"
).configure_legend(
    labelColor="white", titleColor="white", fillColor="#222", strokeColor="#444",
    padding=8, cornerRadius=6
).configure_title(color="white")

/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/c

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting
